# AF2·01 — Folding as a Coevolution Problem

**Mechanism of the day:** the single idea the whole field is built on — a protein's
3D structure leaves a *statistical shadow* in the evolutionary record, and you can
read it off a pile of related sequences before any neural network is involved.

Welcome to the AlphaFold2 ladder. Before we touch the Evoformer or invariant point
attention, we need to understand **what problem AF2 is actually solving and what it
gets to look at.** AlphaFold does not read one sequence. It reads a **multiple
sequence alignment (MSA)** — the query protein stacked with hundreds of its
evolutionary cousins — and that changes everything.

Here is why. If residues `i` and `j` touch in the folded structure, they are under
a joint constraint: a mutation at `i` that would break the contact tends to be
compensated by a mutation at `j`. Over evolutionary time the two columns of the MSA
therefore **coevolve** — their residues are statistically correlated. Flip it
around: *find the correlated column pairs and you have found the contacts.* That is
structure information, extracted from sequences alone.

This notebook builds that pipeline from scratch on a toy MSA where we plant the
contacts ourselves, so we know the answers:

1. measure coevolution with **mutual information**, and recover contacts,
2. hit the two problems that make raw coevolution insufficient —
   **phylogenetic background** (fixed by the *average product correction*) and
   **indirect couplings** (the transitivity trap),
3. name the two objects AF2 will carry from here to the end: the **MSA
   representation** and the **pair representation**.

That last problem — indirect couplings — is exactly the one the Evoformer's
triangle operations exist to solve (rung 03). So this notebook is not just
background; it sets up the puzzle the entire trunk is built to crack.

**How to use this notebook:** read the concept, implement the reps (they
`raise NotImplementedError`), make the checkpoints (`assert`s) pass. Solutions at
the very bottom. Pure numpy — runs instantly on a laptop.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

rng = np.random.default_rng(0)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
BLUE, GREEN, INK = '#2a78d6', '#008300', '#52514e'

AA = 'ACDEFGHIKLMNPQRSTVWY'
Q = len(AA)          # 20 amino acids
L = 48               # residues in our toy protein
N = 1500             # sequences in the MSA
print('toy MSA target:', N, 'sequences x', L, 'residues, alphabet', Q)

## Part 1 — a toy MSA with a known structure

We synthesize the MSA so we know the ground truth. Three ingredients, each mirroring
something real:

- **Per-column conservation.** Every position has its own amino-acid preference
  (some columns picky, some permissive), just like real protein families.
- **Planted contacts.** We choose a set of residue pairs `(i, j)` to be "in contact"
  and make them **coevolve**: at each such pair, residue `j` is a fixed function of
  residue `i` most of the time. These correlated columns are the signal we want to
  recover — they stand in for spatial contacts.
- **Phylogeny (clades).** Real sequences are not independent draws; they are related
  by a tree. We fake this with four **clades**, each sharing a consensus over a block
  of columns. This injects correlations between columns that are *not* in contact —
  the confound that will trip up naive coevolution and motivate the fix in Part 3.

You are handed the generator (it is data plumbing, not the mechanism). Read it, then
we get to work on the columns it produces.

In [ ]:
# --- planted contacts: pairs that will coevolve. Kept off the clade block below. ---
BLOCK = list(range(0, 14))                           # a contiguous block that carries clade identity
NONBLOCK = [c for c in range(L) if c not in BLOCK]

rng2 = np.random.default_rng(3)
cand = [(i, j) for i in NONBLOCK for j in NONBLOCK if j >= i + 4]
rng2.shuffle(cand)
EDGES, CONTACTS = [], set()
for (i, j) in cand[:16]:
    EDGES.append((i, j, rng2.uniform(0.30, 0.60)))  # coupling strength varies (weak..medium)
    CONTACTS.add((min(i, j), max(i, j)))

# a transitivity chain A -> B -> C: A-B and B-C are contacts, A-C is NOT (Part 3)
A, B, Cc = NONBLOCK[2], NONBLOCK[10], NONBLOCK[18]
EDGES += [(A, B, 0.9), (B, Cc, 0.9)]
CONTACTS |= {(min(A, B), max(A, B)), (min(B, Cc), max(B, Cc))}

PERM = {(i, j): rng.permutation(Q) for (i, j, _) in EDGES}   # the coupling rule per contact
PROF = np.array([rng.dirichlet(np.ones(Q) * 0.8) for _ in range(L)])   # per-column preference
NCLADE = 4
CONSENSUS = [{c: int(rng.integers(Q)) for c in BLOCK} for _ in range(NCLADE)]

def generate_msa(N):
    '''Toy MSA: conservation + planted coevolving contacts + phylogenetic clades.'''
    msa = np.zeros((N, L), dtype=int)
    for p in range(L):                              # 1) draw each column from its preference
        msa[:, p] = rng.choice(Q, size=N, p=PROF[p])
    clade = rng.integers(NCLADE, size=N)            # 2) phylogeny: clade consensus on the block
    for c in BLOCK:
        for g in range(NCLADE):
            hit = (clade == g) & (rng.random(N) < 0.9)
            msa[hit, c] = CONSENSUS[g][c]
    for (i, j, k) in sorted(EDGES, key=lambda e: (e[0], e[1])):   # 3) couple contacts (src->tgt)
        cp = rng.random(N) < k
        msa[cp, j] = PERM[(i, j)][msa[cp, i]]
    return msa

msa = generate_msa(N)
TRUE = np.zeros((L, L), dtype=bool)
for (i, j) in CONTACTS:
    TRUE[i, j] = TRUE[j, i] = True

def decode(row):
    return ''.join(AA[i] for i in row)

print(len(CONTACTS), 'planted contacts | 4 clades | first two sequences:')
print(' ', decode(msa[0]))
print(' ', decode(msa[1]))

fig, ax = plt.subplots(figsize=(9, 3.4))
ax.imshow(msa[:80], aspect='auto', interpolation='nearest', cmap='tab20')
ax.set_xlabel('residue position'); ax.set_ylabel('sequence in MSA')
ax.set_title('the toy MSA (first 80 sequences) — each colour is an amino acid')
plt.show()
print('Vertical bands = conserved columns. Some column PAIRS move together — that')
print('coevolution, planted at the contacts, is the signal we are about to extract.')

### Rep 1 — `profile(column)`
Warm-up: the per-column amino-acid distribution (its "profile"), the most basic MSA
feature. Given a column — an integer array of length `N` — return the length-`Q`
vector of frequencies. A **conserved** column has its mass on one or two residues; a
**variable** column is spread out. AF2's MSA representation is a learned generalization
of exactly this.

In [ ]:
def profile(column):
    '''Amino-acid frequency vector [Q] for one MSA column (int array [N]).'''
    # YOUR CODE HERE
    # hint: np.bincount(column, minlength=Q) / len(column)
    raise NotImplementedError

# --- checkpoint ---
p0 = profile(msa[:, 0])
assert p0.shape == (Q,) and np.isclose(p0.sum(), 1.0), 'profile must be a distribution over Q'
# a clade-block column is more conserved (peakier) than a typical free column
peak_block = max(profile(msa[:, c]).max() for c in BLOCK[:4])
peak_free  = np.mean([profile(msa[:, c]).max() for c in NONBLOCK[-6:]])
assert peak_block > peak_free, 'block columns should look more conserved'
print('profile ok — most conserved block column peaks at %.2f, typical free col ~%.2f'
      % (peak_block, peak_free))

### Rep 2 — `mutual_information(col_i, col_j)`
The coevolution measure. Mutual information asks: **how much does knowing the residue
at `i` tell you about the residue at `j`?** Independent columns → 0. Perfectly
coupled columns → large.

$$
\mathrm{MI}(i, j) = \sum_{a, b} f_{ij}(a, b)\,\log\!\frac{f_{ij}(a, b)}{f_i(a)\,f_j(b)}
$$

where `f_i` is the single-column profile and `f_ij` the joint frequency of pair
`(a, b)`. Use a small **pseudocount** so unseen pairs don't produce `log 0` — finite
MSAs never see every combination.

In [ ]:
def mutual_information(col_i, col_j, pseudo=0.5):
    '''MI in nats between two MSA columns (int arrays [N]).'''
    # YOUR CODE HERE
    # hint: one-hot each column [N,Q]; f_ij = (oh_i.T @ oh_j + pseudo/Q) / (N + pseudo)
    #       f_i, f_j from the marginals; MI = sum f_ij * log(f_ij / outer(f_i, f_j))
    raise NotImplementedError

# --- checkpoint ---
mi_indep = mutual_information(msa[:, NONBLOCK[-1]], msa[:, NONBLOCK[-2]])
mi_chain = mutual_information(msa[:, A], msa[:, B])          # a strong planted contact
assert mi_chain > mi_indep, 'a contact must show more MI than an unrelated pair'
assert mi_indep < 0.15, 'unrelated columns should have near-zero MI'
assert mi_chain > 1.0, 'the strong A-B contact should have high MI'
print('MI ok — unrelated pair %.3f nats, strong contact A-B %.3f nats' % (mi_indep, mi_chain))

### Rep 3 — `coevolution_matrix(msa)`
Run `mutual_information` over **every** column pair to get an `[L, L]` symmetric
matrix. This is the classic contact predictor: high entries are candidate contacts.
We will judge it against the planted truth.

(Efficiency note: computing all pairs with explicit loops is fine at `L = 48`. The
one-hot-then-matmul trick vectorizes the joint counts.)

In [ ]:
def coevolution_matrix(msa, pseudo=0.5):
    '''[L, L] symmetric MI matrix over all column pairs (zero diagonal).'''
    # YOUR CODE HERE
    # hint: loop i<j, fill M[i,j]=M[j,i]=mutual_information(msa[:,i], msa[:,j], pseudo)
    raise NotImplementedError

# --- checkpoint ---
MI = coevolution_matrix(msa)
assert MI.shape == (L, L)
assert np.allclose(MI, MI.T), 'MI is symmetric'
assert np.allclose(np.diag(MI), 0), 'diagonal should be zeroed'
print('coevolution matrix ok — shape', MI.shape, '| max off-diagonal %.2f nats' % MI.max())

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2))
axes[0].imshow(MI, cmap='viridis'); axes[0].set_title('mutual information  MI[i, j]', fontsize=10)
yi, xi = np.where(np.triu(TRUE))
axes[1].scatter(xi, yi, s=10, color=GREEN); axes[1].scatter(yi, xi, s=10, color=GREEN)
axes[1].set_xlim(0, L); axes[1].set_ylim(L, 0); axes[1].set_aspect('equal')
axes[1].set_title('true contacts (what we planted)', fontsize=10)
for ax in axes: ax.set_xlabel('residue j'); ax.set_ylabel('residue i')
plt.tight_layout(); plt.show()
print('The MI map has bright spots at contacts — but also a bright BLOCK in the')
print('top-left corner that is not contacts at all. That is phylogeny. Part 2 fixes it.')

### Rep 4 — `precision_at_k(score, true, k)`
To compare predictors we need a number. Rank all pairs (excluding near-diagonal
neighbours, `|i - j| < 4`, which are trivially close) by their score, take the top
`k`, and report the fraction that are real contacts. This "precision@k" is the
standard contact-prediction metric; `k = L` and `k = L/2` are common.

In [ ]:
def precision_at_k(score, true, k, sep=4):
    '''Fraction of the top-k highest-scoring pairs (|i-j|>=sep) that are true contacts.'''
    # YOUR CODE HERE
    # hint: build a list of (score[i,j], true[i,j]) for i<j with j-i>=sep,
    #       sort descending, take k, return mean of the boolean flags
    raise NotImplementedError

# --- checkpoint ---
n_contacts = int(TRUE.sum() // 2)
n_eligible = sum(1 for i in range(L) for j in range(i + 4, L))
random_baseline = n_contacts / n_eligible
p_raw = precision_at_k(MI, TRUE, n_contacts)
assert 0 <= p_raw <= 1
assert p_raw > random_baseline, 'MI should beat random guessing'
# a perfect oracle scores 1.0
assert precision_at_k(TRUE.astype(float), TRUE, n_contacts) == 1.0, 'oracle sanity check'
print('precision@%d: raw MI %.3f   (random guessing would be %.3f)'
      % (n_contacts, p_raw, random_baseline))
print('Better than chance — but far from great. The phylogenetic block is the problem.')

## Part 2 — problem one: phylogenetic background (and the APC fix)

Look again at that bright block in the MI map. Those columns are correlated not
because they touch in 3D, but because sequences that share a **clade** share
residues across *all* of the block's columns at once. Common ancestry, not contact.
This background signal floods the top of the ranking with false positives.

The classic, cheap fix is the **average product correction (APC)**. The intuition:
a truly background-driven pair `(i, j)` has *high MI to everything*, because the
background couples whole groups of columns. So estimate each column's average MI,
and subtract off the part of `MI[i, j]` explained by `i` and `j` both being
generally "hot":

$$
\mathrm{APC}[i, j] = \mathrm{MI}[i, j] - \frac{\mathrm{MI}_i\,\mathrm{MI}_j}{\mathrm{MI}_{\text{mean}}}
$$

where `MI_i` is column `i`'s mean MI to all others and `MI_mean` is the global mean.
This is a rank-one estimate of the background, subtracted out — and it is remarkably
effective.

### Rep 5 — `apc(M)`
Given an MI matrix (zero diagonal), return the APC-corrected matrix. Compute each
column's mean over the off-diagonal entries and the global off-diagonal mean, then
subtract the outer product over the mean. Re-zero the diagonal at the end.

In [ ]:
def apc(M):
    '''Average product correction of an [L, L] coevolution matrix.'''
    # YOUR CODE HERE
    # hint: m = M with diagonal zeroed; col = m.sum(1, keepdims=True) / (L - 1)
    #       allm = mean of off-diagonal entries; corrected = m - (col @ col.T) / allm
    raise NotImplementedError

# --- checkpoint ---
APC = apc(MI)
p_apc = precision_at_k(APC, TRUE, n_contacts)
assert p_apc > p_raw + 0.2, 'APC should sharply improve contact precision'
# count how many phylogenetic block-pairs pollute the raw vs corrected top-k
def block_pairs_in_top(score, k):
    P = sorted(((score[i, j], (i in BLOCK and j in BLOCK))
                for i in range(L) for j in range(i + 4, L)), reverse=True)
    return sum(b for _, b in P[:k])
raw_blk, apc_blk = block_pairs_in_top(MI, n_contacts), block_pairs_in_top(APC, n_contacts)
print('precision@%d:  raw MI %.3f  ->  APC %.3f' % (n_contacts, p_raw, p_apc))
print('phylogenetic (block) false-positives in the top %d:  raw %d  ->  APC %d'
      % (n_contacts, raw_blk, apc_blk))

fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.2))
axes[0].imshow(MI, cmap='viridis'); axes[0].set_title('raw MI (phylogeny dominates)', fontsize=10)
axes[1].imshow(APC, cmap='viridis'); axes[1].set_title('after APC (contacts emerge)', fontsize=10)
for ax in axes: ax.set_xlabel('residue j'); ax.set_ylabel('residue i')
plt.tight_layout(); plt.show()
print('One rank-one subtraction turned a mediocre predictor into a good one. ✓')

## Part 3 — problem two: indirect couplings (the transitivity trap)

APC cleaned up the background, but a deeper problem remains, and **no amount of
pairwise statistics can fix it.** We planted a chain: `A` contacts `B`, and `B`
contacts `C` — but `A` and `C` do **not** touch. What does coevolution see?

Because `A` drives `B` and `B` drives `C`, the residues at `A` and `C` end up
correlated *through* `B`. A pairwise method has no way to tell this **indirect**
coupling apart from a real contact. It will confidently report a contact between `A`
and `C` that does not exist.

In [ ]:
noncontact_mi = np.array([MI[i, j] for i in range(L) for j in range(i + 4, L) if not TRUE[i, j]])
print('the planted chain:   A=%d  B=%d  C=%d   (A-C is NOT a contact)' % (A, B, Cc))
print('  MI(A,B) = %.2f   MI(B,C) = %.2f   [both real contacts]' % (MI[A, B], MI[B, Cc]))
print('  MI(A,C) = %.2f   <- spurious! yet it towers over the typical' % MI[A, Cc])
print('           non-contact pair (mean %.2f, 90th pct %.2f)'
      % (noncontact_mi.mean(), np.percentile(noncontact_mi, 90)))
print('  APC(A,C) = %.2f   <- APC does NOT remove it (it is not a background effect)'
      % APC[A, Cc])

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.hist(noncontact_mi, bins=30, color='#cbd5e1', edgecolor='white', label='non-contact pairs')
ax.axvline(MI[A, Cc], color=BLUE, lw=2, label='MI(A,C), the indirect pair')
ax.axvline(MI[A, B], color=GREEN, lw=2, label='MI(A,B), a real contact')
ax.set_xlabel('mutual information (nats)'); ax.set_ylabel('count')
ax.set_title('an indirect coupling masquerading as a contact'); ax.legend(frameon=False, fontsize=8)
plt.show()

### The fix that this whole ladder is about

Distinguishing **direct** from **indirect** coupling requires reasoning about all
positions *jointly*, not pair by pair. The pre-deep-learning answer was **direct
coupling analysis (DCA)** — fit a global statistical model (a Potts model) and read
off its *direct* couplings, which suppresses `A–C` because the `A–C` correlation is
fully explained once `A–B` and `B–C` are in the model.

AlphaFold2's answer is the deep-learning descendant of that idea. Its **triangle
operations** (rung 03) force the pair representation to obey a geometric
consistency rule: if the network believes in edges `A–B` and `B–C`, the edge `A–C`
is constrained to be *compatible with the triangle they form*, rather than just
"also correlated". That is precisely how it untangles indirect from direct. Keep the
`A–B–C` triangle in your mind's eye — you will meet it again as literal matrix
math.

## Part 4 — the two representations AF2 carries everywhere

Everything above lives in two objects, and **every** remaining notebook in this
series updates one or both of them. Name them now:

- **MSA representation** `m[s, i]` — shape `[N_seq, L, c_m]`. A learned feature
  vector per (sequence, residue). Its ancestor is the one-hot MSA / column profile
  you built in Rep 1. The Evoformer refines it with axial attention (rung 02).

- **Pair representation** `z[i, j]` — shape `[L, L, c_z]`. A learned feature vector
  per residue pair. Its ancestor is the coevolution matrix you built in Rep 3 —
  literally "what do we believe about the relationship between `i` and `j`". The
  triangle operations refine it (rung 03), and the structure module (rungs 05–07)
  reads geometry out of it.

The hand-off between them is the heart of the Evoformer: the MSA informs the pair
representation (which pairs coevolve?), and the pair representation biases how the
MSA attends (which residues should compare notes?). You just built the `c = 1`,
untrained version of both — a raw MSA and an MI matrix. The rest of the ladder makes
them learned, multi-channel, and geometrically consistent.

In [ ]:
# A cartoon of the shapes you will carry from here to the end.
c_m, c_z = 8, 4          # tiny channel counts, for illustration
m_repr = np.zeros((N, L, c_m))
z_repr = np.zeros((L, L, c_z))
# channel 0 of z, initialized from the coevolution signal you just computed — this is
# genuinely how you might seed the pair representation from the input MSA statistics.
z_repr[:, :, 0] = apc(coevolution_matrix(msa))
print('MSA representation m:  shape', m_repr.shape, '  (ancestor: the raw MSA / profiles)')
print('pair representation z: shape', z_repr.shape, '  (ancestor: the APC coevolution matrix)')
print('\nchannel 0 of z, seeded from APC coevolution:')
plt.figure(figsize=(4.2, 3.8))
plt.imshow(z_repr[:, :, 0], cmap='viridis'); plt.colorbar(label='APC coevolution')
plt.title('z[:, :, 0] — the pair rep, notebook 1 version'); plt.xlabel('j'); plt.ylabel('i')
plt.show()

## Reflection — what just transferred

- **AlphaFold reads an MSA, not a sequence**, because structure lives in the
  *correlations between homologous sequences*. Coevolving columns are contacts.
- **Mutual information** turns that idea into a number, and even this crude predictor
  beats chance — the signal is really there in the statistics.
- **Phylogenetic background** (shared ancestry) is a big confound; the **average
  product correction** removes a rank-one estimate of it and sharply improves
  precision. AF2 learns to do this kind of correction, but the problem is the same.
- **Indirect couplings** are the deeper problem, and they are *unfixable* by any
  pairwise method: `A–B` and `B–C` manufacture a phantom `A–C`. Untangling direct
  from indirect requires joint, global reasoning — historically DCA, and in AF2 the
  **triangle operations** you will build in rung 03.
- **Two representations** — the MSA representation `[N,L,c]` and the pair
  representation `[L,L,c]` — are the state that flows through the entire model. You
  built the `c=1` untrained versions today: a raw MSA and a coevolution matrix.

**Next rung:** `AF2·02 — Axial attention over the MSA`. We stop hand-computing
statistics and let a network read the MSA, with attention factorised across its two
axes (compare residues within a sequence; compare sequences at a residue) — and with
the pair representation whispering to it about which residues already look coupled.

---
Scroll down only after you've done the reps.

## Solutions appendix (peek only after trying)

In [ ]:
def profile(column):
    return np.bincount(column, minlength=Q) / len(column)

def mutual_information(col_i, col_j, pseudo=0.5):
    n = len(col_i)
    oh_i, oh_j = np.eye(Q)[col_i], np.eye(Q)[col_j]
    f_ij = (oh_i.T @ oh_j + pseudo / Q) / (n + pseudo)      # [Q, Q] joint, smoothed
    f_i, f_j = f_ij.sum(1), f_ij.sum(0)                     # marginals of the smoothed joint
    return float((f_ij * np.log(f_ij / (np.outer(f_i, f_j) + 1e-12) + 1e-12)).sum())

def coevolution_matrix(msa, pseudo=0.5):
    L = msa.shape[1]
    M = np.zeros((L, L))
    for i in range(L):
        for j in range(i + 1, L):
            M[i, j] = M[j, i] = mutual_information(msa[:, i], msa[:, j], pseudo)
    return M

def precision_at_k(score, true, k, sep=4):
    pairs = [(score[i, j], true[i, j])
             for i in range(len(score)) for j in range(i + sep, len(score))]
    pairs.sort(reverse=True)
    top = pairs[:k]
    return sum(bool(t) for _, t in top) / len(top)

def apc(M):
    L = len(M)
    m = M.copy(); np.fill_diagonal(m, 0)
    col = m.sum(1, keepdims=True) / (L - 1)                 # each column's mean MI
    allm = m[~np.eye(L, dtype=bool)].mean()                 # global off-diagonal mean
    corrected = m - (col @ col.T) / allm                    # subtract rank-one background
    np.fill_diagonal(corrected, 0)
    return corrected

print('reference solutions loaded — re-run the checkpoint cells above')